# EcoHome Energy Advisor - RAG Setup

In this notebook, you'll set up the Retrieval-Augmented Generation (RAG) pipeline for the EcoHome Energy Advisor. This will allow the agent to access and cite relevant energy-saving tips and best practices.

## Learning Objectives
- Set up ChromaDB vector store
- Load and process energy-saving documents
- Create embeddings for document chunks
- Implement semantic search functionality
- Test the RAG pipeline

## Documents Available
- `tip_device_best_practices.txt` - Device-specific optimization tips
- `tip_energy_savings.txt` - General energy-saving strategies
- `tip_energy_storage.txt` - Home battery optimization guide
- `tip_hvac_optimization.txt` - HVAC system efficiency strategies
- `tip_renewable_integration.txt` - Solar panel integration tips
- `tip_seasonal_energy_management.txt` - Seasonal optimization strategies
- `tip_smart_home_automation.txt` - Smart home automation tips


## 1. Import Required Libraries


In [1]:
# Import the necessary libraries for RAG setup
import os
from langchain_chroma  import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

## 2. Load and Process Documents


In [3]:
# Load the energy-saving tip documents
# Load all documents from the data/documents directory
# Use TextLoader to load the documents

documents = []
document_paths = [
    "data/documents/tip_device_best_practices.txt",
    "data/documents/tip_energy_savings.txt",
    "data/documents/tip_energy_storage.txt",
    "data/documents/tip_hvac_optimization.txt",
    "data/documents/tip_renewable_integration.txt",
    "data/documents/tip_seasonal_energy_management.txt",
    "data/documents/tip_smart_home_automation.txt"
]

for doc_path in document_paths:
    if os.path.exists(doc_path):
        loader = TextLoader(doc_path)
        docs = loader.load()
        documents.extend(docs)
        print(f"Loaded {len(docs)} documents from {doc_path}")
    else:
        print(f"Warning: {doc_path} not found")

print(f"Total documents loaded: {len(documents)}")


Loaded 1 documents from data/documents/tip_device_best_practices.txt
Loaded 1 documents from data/documents/tip_energy_savings.txt
Loaded 1 documents from data/documents/tip_energy_storage.txt
Loaded 1 documents from data/documents/tip_hvac_optimization.txt
Loaded 1 documents from data/documents/tip_renewable_integration.txt
Loaded 1 documents from data/documents/tip_seasonal_energy_management.txt
Loaded 1 documents from data/documents/tip_smart_home_automation.txt
Total documents loaded: 7


## 3. Split Documents into Chunks


In [4]:
# Split documents into smaller chunks for better retrieval
# Use RecursiveCharacterTextSplitter with appropriate chunk_size and chunk_overlap
# Experiment with different chunk sizes (e.g., 500, 1000, 1500 characters)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

# Split the documents
splits = text_splitter.split_documents(documents)
print(f"Split {len(documents)} documents into {len(splits)} chunks")

# Show sample chunk
if splits:
    print(f"\nSample chunk (first 200 characters):")
    print(splits[0].page_content[:200] + "...")


Split 7 documents into 31 chunks

Sample chunk (first 200 characters):
Device-Specific Best Practices for Energy Optimization

Large devices like electric vehicles, washing machines, and dishwashers often support delayed start or timer functions. This creates opportuniti...


## 4. Create Vector Store


In [5]:
# Create a ChromaDB vector store
# Initialize OpenAIEmbeddings
# Create the vector store with the document chunks
# Persist the vector store to disk for future use

# Set up the persist directory
persist_directory = "data/vectorstore"
os.makedirs(persist_directory, exist_ok=True)

# Initialize embeddings
embeddings = OpenAIEmbeddings(
    base_url="https://openai.vocareum.com/v1",
    api_key=os.getenv("VOCAREUM_API_KEY")
)

# Create the vector store
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory=persist_directory
)

print(f"Vector store created and persisted to {persist_directory}")
print(f"Total vectors stored: {len(splits)}")

Vector store created and persisted to data/vectorstore
Total vectors stored: 31


## 5. Test the RAG Pipeline


In [6]:
# Test the search functionality
# Try different queries related to energy optimization
# Test queries like:
# - "electric vehicle charging tips"
# - "thermostat optimization"
# - "dishwasher energy saving"
# - "solar power maximization"

test_queries = [
    "electric vehicle charging tips",
    "thermostat optimization",
    "dishwasher energy saving",
    "solar power maximization",
    "HVAC system efficiency",
    "pool pump scheduling"
]

print("=== Testing Vector Search ===")
for query in test_queries:
    print(f"\nQuery: '{query}'")
    docs = vectorstore.similarity_search(query, k=2)
    for i, doc in enumerate(docs):
        print(f"  Result {i+1}: {doc.page_content[:100]}...")


=== Testing Vector Search ===

Query: 'electric vehicle charging tips'
  Result 1: Device-Specific Best Practices for Energy Optimization

Large devices like electric vehicles, washin...
  Result 2: EV CHARGING LOAD STAGGERING:
- Stagger EV charging start times to avoid coincident peaks with HVAC o...

Query: 'thermostat optimization'
  Result 1: Device-Specific Best Practices for Energy Optimization

Large devices like electric vehicles, washin...
  Result 2: EV CHARGING LOAD STAGGERING:
- Stagger EV charging start times to avoid coincident peaks with HVAC o...

Query: 'thermostat optimization'
  Result 1: THERMOSTAT SETPOINT OPTIMIZATION:
- Winter heating: 68°F during day, 62-65°F during sleep/away perio...
  Result 2: Seasonal Energy Management Guide for Year-Round Optimization

WINTER ENERGY OPTIMIZATION (December -...

Query: 'dishwasher energy saving'
  Result 1: THERMOSTAT SETPOINT OPTIMIZATION:
- Winter heating: 68°F during day, 62-65°F during sleep/away perio...
  Result 2: Se

## 6. Test the Search Tool


In [7]:
# Test the search_energy_tips tool from tools.py
# Import and test the tool with various queries
# Verify that it returns relevant results

from tools import search_energy_tips

# Test the search_energy_tips function
print("=== Testing search_energy_tips Tool ===")

test_queries = [
    "electric vehicle charging",
    "thermostat settings",
    "dishwasher optimization",
    "solar power tips"
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    result = search_energy_tips.invoke(
        input={
            "query": query, 
            "max_results": 3,
        }
    )
    
    if "error" in result:
        print(f"  Error: {result['error']}")
    else:
        print(f"  Found {result['total_results']} results")
        for i, tip in enumerate(result['tips']):
            print(f"    {i+1}. {tip['content'][:100]}...")
            print(f"       Source: {tip['source']}")
            print(f"       Relevance: {tip['relevance_score']}")


=== Testing search_energy_tips Tool ===

Query: 'electric vehicle charging'
Using OpenAI API key: voc-... and base URL: https://openai.vocareum.com/v1
  Found 3 results
    1. WATER HEATING EFFICIENCY:
- Reduce hot water usage with low-flow showerheads and faucets
- Hot water...
       Source: data/documents/tip_energy_savings.txt
       Relevance: high
    2. SUMMER ENERGY OPTIMIZATION (June - August):
- Pre-cool home during early morning off-peak hours befo...
       Source: data/documents/tip_seasonal_energy_management.txt
       Relevance: high
    3. THERMOSTAT SETPOINT OPTIMIZATION:
- Winter heating: 68°F during day, 62-65°F during sleep/away perio...
       Source: data/documents/tip_hvac_optimization.txt
       Relevance: medium

Query: 'thermostat settings'
Using OpenAI API key: voc-... and base URL: https://openai.vocareum.com/v1
  Found 3 results
    1. WATER HEATING EFFICIENCY:
- Reduce hot water usage with low-flow showerheads and faucets
- Hot water...
       Source: data